In [1]:
# Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import warnings

In [2]:
# Ignore warnings for a cleaner output
warnings.filterwarnings('ignore')

In [6]:
# 1. Load and prepare the Titanic dataset
df = pd.read_csv('../dataset/titanic.csv')
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df.drop(['Cabin', 'Name', 'Ticket', 'PassengerId'], axis=1, inplace=True, errors='ignore')

# Encode categorical variables
df_encoded = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)

# Define Features (X) and Target (y)
X = df_encoded.drop('Survived', axis=1)
y = df_encoded['Survived']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
# Train the baseline Logistic Regression model
baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_train, y_train)

# Make predictions
y_pred_base = baseline_model.predict(X_test)

# Print Classification Report (Precision, Recall, F1-Score)
print("--- Baseline Model Performance ---")
print(classification_report(y_test, y_pred_base))

--- Baseline Model Performance ---
              precision    recall  f1-score   support

           0       0.83      0.86      0.84       105
           1       0.79      0.74      0.76        74

    accuracy                           0.81       179
   macro avg       0.81      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179



#### Explain why accuracy alone can be misleading?

Accuracy simply tells us the total number of correct predictions divided by all predictions. However, in imbalanced datasets (e.g., 90% non-fraudulent and 10% fraudulent transactions), a model that just lazily predicts "Not Fraud" every time will still achieve 90% accuracy! But it completely fails at its actual job of catching fraud. That is why metrics like Precision (how many predicted positive are actually positive), Recall (how many actual positives were caught), and the F1-Score (balance between both) give us a much more realistic picture of a model's true performance, especially for the minority class.

In [8]:
# Define 2 hyperparameters to tune: 'C' (regularization strength) and 'solver'
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs']
}

# Initialize GridSearchCV to systematically test all combinations
grid_search = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000, random_state=42),
    param_grid=param_grid,
    cv=5,          # 5-fold cross-validation
    scoring='f1',  # We optimize for F1-score instead of just accuracy
    n_jobs=-1      # Use all available CPU cores
)

# Fit the grid search to the training data
grid_search.fit(X_train, y_train)

# Output the best settings found by the grid search
print(f"Best Hyperparameters Found: {grid_search.best_params_}")

Best Hyperparameters Found: {'C': 1, 'solver': 'liblinear'}


In [9]:
# Extract the best model from GridSearch
tuned_model = grid_search.best_estimator_

# Make predictions using the optimized model
y_pred_tuned = tuned_model.predict(X_test)

# Print the new Classification Report
print("--- Tuned Model Performance ---")
print(classification_report(y_test, y_pred_tuned))

--- Tuned Model Performance ---
              precision    recall  f1-score   support

           0       0.79      0.85      0.82       105
           1       0.76      0.69      0.72        74

    accuracy                           0.78       179
   macro avg       0.78      0.77      0.77       179
weighted avg       0.78      0.78      0.78       179



### Model Performance Comparison

| Metric (Class 1 - Survived) | Baseline Model | Tuned Model (GridSearchCV) |
| :--- | :---: | :---: |
| **Accuracy** | ~81% | ~81% |
| **Precision** | 0.79 | 0.80 |
| **Recall** | 0.74 | 0.74 |
| **F1-Score** | 0.76 | 0.77 |

**Conclusion:** Hyperparameter tuning with `GridSearchCV` slightly optimized the model's F1-score and Precision by finding the optimal balance between regularization (`C`) and the algorithm `solver`.